# DistilBERT Router: Two-Head Classifier, Last Layer Unfrozen

This notebook trains a Router that maps a query to two positive integers: search depth and beam width.

Configuration:
- Model: `distilbert-base-uncased`
- Encoder: all DistilBERT layers frozen except the last transformer layer
- Trainable layers: last DistilBERT transformer layer + dropout + depth classifier head + width classifier head
- Max sequence length: 64
- Batch size: 16
- Epochs: 30
- Encoder learning rate: 2e-5
- Head learning rate: 1e-3
- Weight decay: 0.01
- Dropout: 0.2
- Scheduler: none
- Loss: unweighted cross entropy for depth + unweighted cross entropy for width
- Evaluation: 5-fold CV, roughly 80/20 per fold
- Oversampling: none


In [7]:
# If needed, install dependencies first:
# !pip install torch transformers scikit-learn pandas tqdm

import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

In [8]:
CONFIG = {
    "data_path": "/content/clean_router_training_data.jsonl",
    "model_name": "distilbert-base-uncased",
    "max_length": 64,
    "batch_size": 16,
    "epochs": 30,
    "encoder_learning_rate": 2e-5,
    "head_learning_rate": 1e-3,
    "weight_decay": 0.01,
    "dropout": 0.2,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "seed": 42,
}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='cuda')

## Load Data

Expected JSONL format:

```json
{"query": "...", "strategy": "(2,3)"}
```

In [9]:
def parse_strategy(strategy: str) -> tuple[int, int]:
    strategy = strategy.strip()
    if not (strategy.startswith("(") and strategy.endswith(")")):
        raise ValueError(f"Bad strategy format: {strategy}")
    left, right = strategy[1:-1].split(",")
    return int(left), int(right)

data_path = Path(CONFIG["data_path"])
rows = []
with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        depth, width = parse_strategy(obj["strategy"])
        rows.append({
            "query": obj["query"],
            "strategy": obj["strategy"],
            "depth": depth,
            "width": width,
        })

df = pd.DataFrame(rows)

depth_values = sorted(df["depth"].unique())
width_values = sorted(df["width"].unique())
depth2id = {value: idx for idx, value in enumerate(depth_values)}
id2depth = {idx: value for value, idx in depth2id.items()}
width2id = {value: idx for idx, value in enumerate(width_values)}
id2width = {idx: value for value, idx in width2id.items()}

df["depth_label"] = df["depth"].map(depth2id)
df["width_label"] = df["width"].map(width2id)

print(f"Examples: {len(df)}")
print(f"Depth classes: {depth_values}")
print(f"Width classes: {width_values}")
display(df.head())
display(df["strategy"].value_counts().rename_axis("strategy").reset_index(name="count"))
display(df["depth"].value_counts().sort_index().rename_axis("depth").reset_index(name="count"))
display(df["width"].value_counts().sort_index().rename_axis("width").reset_index(name="count"))

Examples: 127
Depth classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Width classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(5)]


,query,strategy,depth,width,depth_label,width_label
0,what continent is australia in,"(1,2)",1,2,0,1
1,where does missouri river end,"(1,1)",1,1,0,0
2,who wrote the book of st. john,"(1,1)",1,1,0,0
3,what illness does michael j fox have,"(1,1)",1,1,0,0
4,what did john howard study at university,"(4,1)",4,1,3,0


,strategy,count
0,"(1,1)",43
1,"(2,1)",20
2,"(1,2)",13
3,"(3,1)",10
4,"(2,3)",7
5,"(2,5)",7
6,"(1,3)",4
7,"(4,1)",4
8,"(1,5)",4
9,"(4,3)",4


,depth,count
0,1,64
1,2,37
2,3,14
3,4,12


,width,count
0,1,77
1,2,18
2,3,17
3,5,15


## Folds

This splitter distributes full strategy labels across folds as evenly as possible, without oversampling or validation leakage. The model predicts depth and width separately.


In [10]:
def make_stratified_like_folds(labels, n_splits: int, seed: int):
    rng = np.random.default_rng(seed)
    label_to_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        label_to_indices[label].append(idx)

    folds = [[] for _ in range(n_splits)]
    for offset, label in enumerate(sorted(label_to_indices)):
        indices = np.array(label_to_indices[label])
        rng.shuffle(indices)
        for position, idx in enumerate(indices):
            folds[(offset + position) % n_splits].append(int(idx))

    all_indices = set(range(len(labels)))
    splits = []
    for val_indices in folds:
        val_indices = sorted(val_indices)
        train_indices = sorted(all_indices - set(val_indices))
        splits.append((train_indices, val_indices))
    return splits

splits = make_stratified_like_folds(df["strategy"].tolist(), CONFIG["n_splits"], CONFIG["seed"])
for fold_id, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"Fold {fold_id}: train={len(train_idx)}, val={len(val_idx)}")
    print("  val strategy distribution:", dict(Counter(df.iloc[val_idx]["strategy"])))
    print("  val depth distribution:", dict(Counter(df.iloc[val_idx]["depth"])))
    print("  val width distribution:", dict(Counter(df.iloc[val_idx]["width"])))

Fold 1: train=101, val=26
  val strategy distribution: {'(1,2)': 2, '(1,1)': 9, '(3,1)': 2, '(1,3)': 1, '(2,1)': 4, '(2,2)': 1, '(4,5)': 1, '(4,1)': 1, '(2,3)': 1, '(1,5)': 1, '(2,5)': 1, '(3,3)': 1, '(4,3)': 1}
  val depth distribution: {1: 13, 3: 3, 2: 7, 4: 3}
  val width distribution: {2: 3, 1: 16, 3: 4, 5: 3}
Fold 2: train=100, val=27
  val strategy distribution: {'(2,3)': 2, '(4,3)': 1, '(1,2)': 3, '(3,5)': 1, '(1,1)': 9, '(1,5)': 1, '(3,1)': 2, '(2,1)': 4, '(2,5)': 1, '(3,3)': 1, '(2,2)': 1, '(4,5)': 1}
  val depth distribution: {2: 8, 4: 2, 1: 13, 3: 4}
  val width distribution: {3: 4, 2: 4, 5: 4, 1: 15}
Fold 3: train=100, val=27
  val strategy distribution: {'(1,1)': 9, '(4,1)': 1, '(3,1)': 2, '(2,3)': 2, '(2,1)': 4, '(1,2)': 3, '(2,5)': 2, '(2,2)': 1, '(1,3)': 1, '(4,5)': 1, '(4,3)': 1}
  val depth distribution: {1: 13, 4: 3, 3: 2, 2: 9}
  val width distribution: {1: 16, 3: 4, 2: 4, 5: 3}
Fold 4: train=103, val=24
  val strategy distribution: {'(1,1)': 8, '(3,1)': 2, '(1,2)':

## Dataset and Model

In [11]:
class RouterDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_length: int):
        self.queries = frame["query"].tolist()
        self.depth_labels = frame["depth_label"].astype(int).tolist()
        self.width_labels = frame["width_label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.queries[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "depth_labels": torch.tensor(self.depth_labels[idx], dtype=torch.long),
            "width_labels": torch.tensor(self.width_labels[idx], dtype=torch.long),
        }


class DistilBertLastLayerTwoHeadRouter(nn.Module):
    def __init__(self, model_name: str, num_depth_labels: int, num_width_labels: int, dropout: float):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        for param in self.encoder.parameters():
            param.requires_grad = False

        for param in self.encoder.transformer.layer[-1].parameters():
            param.requires_grad = True

        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.depth_classifier = nn.Linear(hidden_size, num_depth_labels)
        self.width_classifier = nn.Linear(hidden_size, num_width_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(cls_embedding)
        depth_logits = self.depth_classifier(pooled)
        width_logits = self.width_classifier(pooled)
        return depth_logits, width_logits


## Training and Evaluation Helpers

In [12]:
def train_one_epoch(model, loader, optimizer, depth_loss_fn, width_loss_fn):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        depth_labels = batch["depth_labels"].to(device)
        width_labels = batch["width_labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        depth_logits, width_logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = depth_loss_fn(depth_logits, depth_labels) + width_loss_fn(width_logits, width_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * depth_labels.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, depth_loss_fn, width_loss_fn):
    model.eval()
    total_loss = 0.0
    all_depth_preds = []
    all_width_preds = []
    all_depth_labels = []
    all_width_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        depth_labels = batch["depth_labels"].to(device)
        width_labels = batch["width_labels"].to(device)

        depth_logits, width_logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = depth_loss_fn(depth_logits, depth_labels) + width_loss_fn(width_logits, width_labels)

        depth_preds = depth_logits.argmax(dim=-1)
        width_preds = width_logits.argmax(dim=-1)
        total_loss += loss.item() * depth_labels.size(0)

        all_depth_preds.extend(depth_preds.cpu().tolist())
        all_width_preds.extend(width_preds.cpu().tolist())
        all_depth_labels.extend(depth_labels.cpu().tolist())
        all_width_labels.extend(width_labels.cpu().tolist())

    metrics = compute_metrics(all_depth_labels, all_width_labels, all_depth_preds, all_width_preds)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics, all_depth_labels, all_width_labels, all_depth_preds, all_width_preds


def compute_metrics(depth_labels, width_labels, depth_preds, width_preds):
    gold_depth = [id2depth[i] for i in depth_labels]
    pred_depth = [id2depth[i] for i in depth_preds]
    gold_width = [id2width[i] for i in width_labels]
    pred_width = [id2width[i] for i in width_preds]

    exact = [gd == pd and gw == pw for gd, gw, pd, pw in zip(gold_depth, gold_width, pred_depth, pred_width)]
    gold_strategy = [f"({d},{w})" for d, w in zip(gold_depth, gold_width)]
    pred_strategy = [f"({d},{w})" for d, w in zip(pred_depth, pred_width)]

    return {
        "exact_accuracy": float(np.mean(exact)),
        "depth_accuracy": accuracy_score(gold_depth, pred_depth),
        "width_accuracy": accuracy_score(gold_width, pred_width),
        "depth_macro_f1": f1_score(gold_depth, pred_depth, average="macro", zero_division=0),
        "width_macro_f1": f1_score(gold_width, pred_width, average="macro", zero_division=0),
        "strategy_macro_f1": f1_score(gold_strategy, pred_strategy, average="macro", zero_division=0),
        "depth_abs_error": np.mean([abs(a - b) for a, b in zip(gold_depth, pred_depth)]),
        "width_abs_error": np.mean([abs(a - b) for a, b in zip(gold_width, pred_width)]),
        "cost_sensitive_error": np.mean([
            abs(gd - pd) + 0.5 * abs(gw - pw)
            for gd, gw, pd, pw in zip(gold_depth, gold_width, pred_depth, pred_width)
        ]),
    }

## Majority Baselines

In [13]:
majority_depth = df["depth"].mode()[0]
majority_width = df["width"].mode()[0]
majority_strategy = df["strategy"].mode()[0]

print("Majority depth:", majority_depth)
print("Majority width:", majority_width)
print("Majority full strategy:", majority_strategy)
print("Always majority full strategy exact accuracy:", (df["strategy"] == majority_strategy).mean())
print("Always majority-depth/majority-width exact accuracy:", ((df["depth"] == majority_depth) & (df["width"] == majority_width)).mean())
print("Always majority depth accuracy:", (df["depth"] == majority_depth).mean())
print("Always majority width accuracy:", (df["width"] == majority_width).mean())

Majority depth: 1
Majority width: 1
Majority full strategy: (1,1)
Always majority full strategy exact accuracy: 0.33858267716535434
Always majority-depth/majority-width exact accuracy: 0.33858267716535434
Always majority depth accuracy: 0.5039370078740157
Always majority width accuracy: 0.6062992125984252


## 5-Fold Cross-Validation

In [14]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

fold_results = []
all_predictions = []

for fold_id, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"\n===== Fold {fold_id}/{CONFIG['n_splits']} =====")
    set_seed(CONFIG["seed"] + fold_id)

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    print(f"Train size: {len(train_df)} | Val size: {len(val_df)}")

    train_dataset = RouterDataset(train_df, tokenizer, CONFIG["max_length"])
    val_dataset = RouterDataset(val_df, tokenizer, CONFIG["max_length"])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

    model = DistilBertLastLayerTwoHeadRouter(
        CONFIG["model_name"],
        num_depth_labels=len(depth_values),
        num_width_labels=len(width_values),
        dropout=CONFIG["dropout"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoder.transformer.layer[-1].parameters(), "lr": CONFIG["encoder_learning_rate"]},
            {"params": model.depth_classifier.parameters(), "lr": CONFIG["head_learning_rate"]},
            {"params": model.width_classifier.parameters(), "lr": CONFIG["head_learning_rate"]},
        ],
        weight_decay=CONFIG["weight_decay"],
    )
    trainable_param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable_param_count:,}")
    depth_loss_fn = nn.CrossEntropyLoss()
    width_loss_fn = nn.CrossEntropyLoss()

    best_val_loss = math.inf
    best_state = None
    best_epoch = 0
    patience_used = 0

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, depth_loss_fn, width_loss_fn)
        val_metrics, _, _, _, _ = evaluate(model, val_loader, depth_loss_fn, width_loss_fn)

        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"exact_acc={val_metrics['exact_accuracy']:.3f} | "
            f"depth_acc={val_metrics['depth_accuracy']:.3f} | "
            f"width_acc={val_metrics['width_accuracy']:.3f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_used = 0
        else:
            patience_used += 1
            if patience_used >= CONFIG["early_stopping_patience"]:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    final_metrics, depth_labels, width_labels, depth_preds, width_preds = evaluate(
        model, val_loader, depth_loss_fn, width_loss_fn
    )
    final_metrics["fold"] = fold_id
    final_metrics["best_epoch"] = best_epoch
    fold_results.append(final_metrics)

    for row, depth_gold_id, width_gold_id, depth_pred_id, width_pred_id in zip(
        val_df.to_dict("records"), depth_labels, width_labels, depth_preds, width_preds
    ):
        gold_depth = id2depth[depth_gold_id]
        gold_width = id2width[width_gold_id]
        pred_depth = id2depth[depth_pred_id]
        pred_width = id2width[width_pred_id]
        all_predictions.append({
            "fold": fold_id,
            "query": row["query"],
            "gold_strategy": f"({gold_depth},{gold_width})",
            "pred_strategy": f"({pred_depth},{pred_width})",
            "gold_depth": gold_depth,
            "pred_depth": pred_depth,
            "gold_width": gold_width,
            "pred_width": pred_width,
            "correct": gold_depth == pred_depth and gold_width == pred_width,
        })

results_df = pd.DataFrame(fold_results)
predictions_df = pd.DataFrame(all_predictions)
display(results_df)


===== Fold 1/5 =====
Train size: 101 | Val size: 26


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 7,094,024
Epoch 01 | train_loss=2.7732 | val_loss=2.3249 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 02 | train_loss=2.2770 | val_loss=2.3934 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 03 | train_loss=2.2579 | val_loss=2.3472 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 04 | train_loss=2.1613 | val_loss=2.3134 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 05 | train_loss=2.1111 | val_loss=2.3077 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 06 | train_loss=2.0427 | val_loss=2.2820 | exact_acc=0.385 | depth_acc=0.538 | width_acc=0.615
Epoch 07 | train_loss=1.9923 | val_loss=2.2893 | exact_acc=0.385 | depth_acc=0.500 | width_acc=0.615
Epoch 08 | train_loss=1.9499 | val_loss=2.2756 | exact_acc=0.385 | depth_acc=0.500 | width_acc=0.615
Epoch 09 | train_loss=1.8703 | val_loss=2.2931 | exact_acc=0.385 | depth_acc=0.500 | width_acc=0.615
Epoch 10 | train_loss=1.8000 | val_loss=2.2899 | exact_acc=

...

Trainable parameters: 7,094,024
Epoch 01 | train_loss=2.5163 | val_loss=2.1606 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 02 | train_loss=2.3409 | val_loss=2.1753 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 03 | train_loss=2.3009 | val_loss=2.1542 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 04 | train_loss=2.1670 | val_loss=2.1418 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 05 | train_loss=2.1113 | val_loss=2.1307 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 06 | train_loss=2.0610 | val_loss=2.1223 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 07 | train_loss=1.9964 | val_loss=2.1223 | exact_acc=0.292 | depth_acc=0.500 | width_acc=0.625
Epoch 08 | train_loss=1.9340 | val_loss=2.1339 | exact_acc=0.292 | depth_acc=0.500 | width_acc=0.625
Epoch 09 | train_loss=1.8559 | val_loss=2.1406 | exact_acc=0.292 | depth_acc=0.500 | width_acc=0.625
Epoch 10 | train_loss=1.7816 | val_loss=2.1612 | exact_acc=

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 7,094,024
Epoch 01 | train_loss=2.5619 | val_loss=2.2133 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 02 | train_loss=2.2788 | val_loss=2.2418 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 03 | train_loss=2.2667 | val_loss=2.2414 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 04 | train_loss=2.1669 | val_loss=2.2530 | exact_acc=0.348 | depth_acc=0.565 | width_acc=0.652
Epoch 05 | train_loss=2.1126 | val_loss=2.2405 | exact_acc=0.304 | depth_acc=0.478 | width_acc=0.652
Epoch 06 | train_loss=2.0820 | val_loss=2.2226 | exact_acc=0.304 | depth_acc=0.478 | width_acc=0.652
Early stopping at epoch 6.


,exact_accuracy,depth_accuracy,width_accuracy,depth_macro_f1,width_macro_f1,strategy_macro_f1,depth_abs_error,width_abs_error,cost_sensitive_error,loss,fold,best_epoch
0,0.384615,0.500000,0.615385,0.245098,0.190476,0.075214,0.807692,0.884615,1.250000,2.275601,1,8
1,0.370370,0.518519,0.555556,0.221053,0.178571,0.071895,0.740741,1.037037,1.259259,2.362186,2,5
2,0.333333,0.481481,0.592593,0.162500,0.186047,0.045455,0.814815,0.888889,1.259259,2.300562,3,4
3,0.291667,0.500000,0.625000,0.166667,0.192308,0.045161,0.750000,0.833333,1.166667,2.122254,4,7
4,0.347826,0.521739,0.652174,0.171429,0.197368,0.046921,0.782609,0.739130,1.152174,2.213336,5,1


## Cross-Validation Summary

In [15]:
metric_cols = [
    "exact_accuracy",
    "depth_accuracy",
    "width_accuracy",
    "depth_macro_f1",
    "width_macro_f1",
    "strategy_macro_f1",
    "depth_abs_error",
    "width_abs_error",
    "cost_sensitive_error",
    "loss",
]

summary = pd.DataFrame({
    "mean": results_df[metric_cols].mean(),
    "std": results_df[metric_cols].std(ddof=1),
})
display(summary)

results_df.to_csv("two_head_unfreeze_last1_no_oversampling_cv_results.csv", index=False)
predictions_df.to_csv("two_head_unfreeze_last1_no_oversampling_cv_predictions.csv", index=False)

print("Saved fold metrics to two_head_unfreeze_last1_no_oversampling_cv_results.csv")
print("Saved validation predictions to two_head_unfreeze_last1_no_oversampling_cv_predictions.csv")

,mean,std
exact_accuracy,0.345562,0.036055
depth_accuracy,0.504348,0.016309
width_accuracy,0.608141,0.036341
depth_macro_f1,0.193349,0.037382
width_macro_f1,0.188954,0.007082
strategy_macro_f1,0.056929,0.015237
depth_abs_error,0.779171,0.033255
width_abs_error,0.876601,0.108080
cost_sensitive_error,1.217472,0.053375
loss,2.254788,0.091305


Saved fold metrics to two_head_unfreeze_last1_no_oversampling_cv_results.csv
Saved validation predictions to two_head_unfreeze_last1_no_oversampling_cv_predictions.csv


## Inspect Predictions

In [16]:
display(predictions_df.head(30))

display(
    predictions_df.groupby(["gold_strategy", "pred_strategy"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

display(
    predictions_df.groupby(["gold_depth", "pred_depth"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(
    predictions_df.groupby(["gold_width", "pred_width"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)


,fold,query,gold_strategy,pred_strategy,gold_depth,pred_depth,gold_width,pred_width,correct
0,1,what continent is australia in,"(1,2)","(1,1)",1,1,2,1,False
1,1,who wrote the book of st. john,"(1,1)","(1,1)",1,1,1,1,True
2,1,what county is rihanna from,"(1,1)","(1,1)",1,1,1,1,True
3,1,who is jojo simmons mother,"(1,1)","(2,1)",1,2,1,1,False
4,1,who created the character of sherlock holmes,"(1,1)","(1,1)",1,1,1,1,True
5,1,who plays lola bunny on the looney tunes show,"(3,1)","(2,1)",3,2,1,1,False
6,1,what instruments does john williams use,"(1,3)","(1,1)",1,1,3,1,False
7,1,where did robin gibb die,"(1,1)","(1,1)",1,1,1,1,True
8,1,who played daniel larusso,"(2,1)","(2,1)",2,2,1,1,True
9,1,who was king or queen after victoria,"(2,1)","(1,1)",2,1,1,1,False


,gold_strategy,pred_strategy,count
0,"(1,1)","(1,1)",41
6,"(2,1)","(1,1)",17
2,"(1,2)","(1,1)",13
11,"(3,1)","(1,1)",8
9,"(2,3)","(1,1)",7
10,"(2,5)","(1,1)",7
16,"(4,1)","(1,1)",4
3,"(1,3)","(1,1)",4
18,"(4,3)","(1,1)",4
19,"(4,5)","(1,1)",3


,gold_depth,pred_depth,count
0,1,1,61
2,2,1,34
4,3,1,12
6,4,1,12
1,1,2,3
3,2,2,3
5,3,2,2


,gold_width,pred_width,count
0,1,1,77
1,2,1,18
2,3,1,17
3,5,1,15
